In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

env_path = '../.env'
load_dotenv(dotenv_path=env_path)

api_key = os.getenv("GOOGLE_API_KEY")
if api_key:
    chat = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        temperature=0.7,
        max_output_tokens=500,
        top_k = 40,
    )
    print("GoogleGenerativeAI LLM initialized.")
else:
    print("GOOGLE_GENAI_API_KEY not found in environment variables.")

GoogleGenerativeAI LLM initialized.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import Literal


class Feedback(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str


model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
structured_model = model.with_structured_output(
    schema=Feedback.model_json_schema(), method="json_schema"
)

response = structured_model.invoke("The new UI is great!")
print(type(response))  # Feedback
print(response)  # "positive"
  # "The user expresses positive..."

<class 'dict'>
{'sentiment': 'positive', 'summary': 'The user finds the new UI great.'}


In [ ]:
# https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai#structured-output-methods
from datetime import datetime
from pydantic import BaseModel, Field

class convert_to_datetime(BaseModel):
    value: datetime = Field(
        description="Datetime in ISO 8601 format with timezone if available"
    )

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
structured_model = model.with_structured_output(
    schema=convert_to_datetime.model_json_schema(), method="json_schema"
)

response = structured_model.invoke("Schedule a meeting on March 15, 2024 at 3 PM PST.")
print(type(response["value"]))  # ParsedDateTime
print(response)  # 2024-03-15 15:00:00

<class 'str'>
{'value': '2024-03-15T15:00:00-08:00'}


In [10]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Scientist(BaseModel):
    name: str = Field(description="Full name of the scientist")
    field: str = Field(description="Field of study or expertise")
    notable_work: str = Field(description="A brief description of their notable work")
    discoveries: list = Field(description="A list of significant discoveries or contributions made by the scientist")

output_parser = PydanticOutputParser(pydantic_object=Scientist)
# output_parser.get_format_instructions()

human_msg_template = HumanMessagePromptTemplate.from_template("{request}\n{format_instructions}")

chat_prompt = ChatPromptTemplate.from_messages([human_msg_template])
prompt = chat_prompt.format_prompt(
    request="Tell me about Marie Curie.",
    format_instructions=output_parser.get_format_instructions()
).to_messages()

response = chat.invoke(prompt)
sci = output_parser.parse(response.content)
print(sci)  # Scientist

name='Marie Curie' field='Physics and Chemistry' notable_work='Pioneering research on radioactivity, discovery of polonium and radium, and development of mobile X-ray units during World War I.' discoveries=['Polonium', 'Radium', 'Theory of radioactivity', 'Techniques for isolating radioactive isotopes']


In [15]:
from langchain_core.prompts import PromptTemplate

template = "Tell me a joke about {topic}."
prompt = PromptTemplate.from_template(template)

# This works!
prompt.save("prompt.json")

# Load it back
from langchain_core.prompts import load_prompt
loaded = load_prompt("prompt.json")
print(loaded.format(topic="computers"))

Tell me a joke about computers.
